# Jev: pilot comparison, then independent holdout

This notebook tests **100 manually labeled pilot reviews**, using eight questions, and saves Jev results separately from GPT. Nothing sends paid requests unless you explicitly turn on a start switch.

Run sections **1–3** to set up; **4** starts the paid pilot; **5–6** evaluate it. After refining pilot rules, freeze them in **7**, manually complete the separate holdout labels in **8**, then run/evaluate the holdout in **9–10**. This notebook has no full-dataset launch cell.

The corrected human reference is a read-only snapshot of the Google Sheets pilot data, fetched 2026-10-26. It is not the older pre-review CSV. If you change those labels later, explicitly export a new reference and reevaluate both models against the same reference.

The model is pinned to `jev-1.14.0`, rather than a moving `jev-latest` alias. We measure performance on your data; vendor speed/cost claims are not used as results.


## 1 — Check Python and install (no inference)
Select a **Python 3.10 or newer** notebook kernel. The TypeSafe SDK requires it. Run this cell once; restart the kernel if prompted after installation.

In [2]:
import sys
assert sys.version_info >= (3, 10), 'Select a Python 3.10+ kernel before installing typesafe-sdk.'
%pip install -U typesafe-sdk pandas pydantic python-dotenv openai


Note: you may need to restart the kernel to use updated packages.


## 2 — Settings and helpers (no API requests)
Keep `TYPESAFE_API_KEY` in `src/.env`; never paste it into notebook cells. Pricing is a configurable estimate in **USD per million tokens**, checked against the model documentation on 2026-10-26. TypeSafe lists $0.042 per million input tokens and free output tokens; actual billed amounts come from your console.

Keep the same version/questions/model to resume. If you revise questions, use a new `EXPERIMENT_VERSION` so earlier results remain intact. Requests are sequential here so per-request latency is easy to interpret; do not equate this with five-worker GPT throughput.


In [5]:
import os, json, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from dotenv import load_dotenv

ROOT = Path('/Users/rejaimes/repos/coffee-project')
DATA = ROOT / 'data' / 'processed'
OUT = DATA / 'jev'
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))
from jev_evaluation import (
    QUALIFIERS, human_check, spec_check, contract_for, predictions,
    run_jev, timing_cost, evaluate, save_reports, freeze_holdout, check_frozen,
)
from labeling_runner import read_jsonl, digest, canonical

load_dotenv(ROOT / 'src' / '.env')
load_dotenv(DATA / '.env')
MODEL = 'jev-1.14.0'
EXPERIMENT_VERSION = 'jev_questions_v1'
QUESTION_PATH = ROOT / 'src' / 'jev_questions_v1.json'
HUMAN_PATH = OUT / 'pilot_human_reference_2026-10-26.csv'
GPT_PATH = DATA / 'pilot_llm_labels_pilot_v2.jsonl'
PILOT_RESULTS = OUT / f'pilot_{EXPERIMENT_VERSION}.jsonl'
HOLDOUT_RESULTS = OUT / f'holdout_{EXPERIMENT_VERSION}.jsonl'
FREEZE_PATH = OUT / f'frozen_{EXPERIMENT_VERSION}.json'
HOLDOUT_INPUTS = OUT / 'holdout_inputs.csv'
HOLDOUT_HUMAN = OUT / 'holdout_human_labels.csv'
INPUT_USD_PER_MILLION = 0.042
OUTPUT_USD_PER_MILLION = 0.0
RUN_LIMIT = 100                 # Set to 5 for a small first smoke test, then 100.
MIN_REQUEST_INTERVAL = 0.5      # Lower only within your TypeSafe rate limits.

specs = json.loads(QUESTION_PATH.read_text())
spec_check(specs)
print('Questions:', list(specs))
print('Question hash:', digest(canonical(specs)))


ValueError: Questions must have exactly the eight feature names.

## 3 — Verify the pilot and separate holdout (local only)
The source's original seed-42 split is verified against the pilot IDs before trusting the saved holdout. Human labels are kept in the evaluation table only; the API receives **review text and questions**, never human labels.


In [15]:
pilot = human_check(pd.read_csv(HUMAN_PATH))
source = pd.read_csv(DATA / 'santa_barbara_data.csv', low_memory=False)
shuffled = source[['review_id', 'business_id', 'text']].sample(frac=1, random_state=42)
assert len(pilot) == 100
human_check(pilot, shuffled.iloc[:100])
holdout_inputs = pd.read_csv(HOLDOUT_INPUTS)
assert len(holdout_inputs) == 200 and holdout_inputs.review_id.is_unique
assert set(holdout_inputs.review_id) == set(shuffled.iloc[100:300].review_id)
assert not set(pilot.review_id) & set(holdout_inputs.review_id)
assert holdout_inputs.set_index('review_id').sort_index().equals(
    shuffled.iloc[100:300].set_index('review_id').sort_index()
)
pilot_contract = contract_for(pilot, specs, MODEL, EXPERIMENT_VERSION, 'pilot')
existing = predictions(PILOT_RESULTS, pilot_contract)
print(f'Pilot: {len(pilot)}; saved Jev predictions: {len(existing)}; holdout: {len(holdout_inputs)}')
print('Human source:', HUMAN_PATH)


ValueError: Human labels do not match the expected review IDs.

## 4 — Run Jev on the pilot (paid; OFF by default)
Change only the first `START_JEV_PILOT` to True when ready. Each review gets one request containing all eight questions. Validated successes are saved immediately and skipped on rerun. Failures are stored separately and remain eligible to retry. No automatic retry hides extra time or charges; 429/authentication errors stop the run so you can inspect and resume later.

Interrupt to stop; rerun sections 2–4 to resume. An in-flight interrupted request might still be billed even if no response was saved. A malformed JSONL tail causes a clear error instead of silently dropping a record. Timing includes network/model time; session wall time also includes writing and pacing.


In [4]:
START_JEV_PILOT = True
if START_JEV_PILOT:
    START_JEV_PILOT = False
    assert os.getenv('TYPESAFE_API_KEY'), 'Add TYPESAFE_API_KEY to src/.env and rerun section 2.'
    from typesafe_sdk import TypeSafeClient, RetryPolicy
    with TypeSafeClient(model=MODEL, retry=RetryPolicy(max_retries=0), timeout=60.0) as client:
        pilot_rows = run_jev(
            pilot, specs, client, MODEL, EXPERIMENT_VERSION, PILOT_RESULTS,
            stage='pilot', min_interval=MIN_REQUEST_INTERVAL, limit=RUN_LIMIT,
        )
else:
    print('No paid requests. Set START_JEV_PILOT=True only when ready.')


pilot: 1/100 saved; this session failures=0
pilot: 2/100 saved; this session failures=0
pilot: 3/100 saved; this session failures=0
pilot: 4/100 saved; this session failures=0
pilot: 5/100 saved; this session failures=0
pilot: 6/100 saved; this session failures=0
pilot: 7/100 saved; this session failures=0
pilot: 8/100 saved; this session failures=0
pilot: 9/100 saved; this session failures=0
pilot: 10/100 saved; this session failures=0
pilot: 11/100 saved; this session failures=0
pilot: 12/100 saved; this session failures=0
pilot: 13/100 saved; this session failures=0
pilot: 14/100 saved; this session failures=0
pilot: 15/100 saved; this session failures=0
pilot: 16/100 saved; this session failures=0
pilot: 17/100 saved; this session failures=0
pilot: 18/100 saved; this session failures=0
pilot: 19/100 saved; this session failures=0
pilot: 20/100 saved; this session failures=0
pilot: 21/100 saved; this session failures=0
pilot: 22/100 saved; this session failures=0
pilot: 23/100 saved

## 5 — Pilot timing, cost, and disagreements
The timing report works even after a partial run. Final accuracy metrics require all 100 predictions; missing/failed reviews cannot silently improve the score. Cost is calculated from returned usage, including failed responses with available usage, not a fixed price per review. Unknown usage is counted explicitly. Billing-console totals may differ.

The metric files include per-qualifier agreement, per-class precision/recall/F1 and human/predicted counts, a 3×3 confusion matrix, and disagreement text. Undefined metrics stay blank. Macro F1 averages classes present in either truth or predictions; inspect negative/positive class support before drawing conclusions. High agreement on mostly-zero labels can be misleading.


In [5]:
pilot_rows = predictions(PILOT_RESULTS, pilot_contract)
pilot_timing = timing_cost(PILOT_RESULTS, INPUT_USD_PER_MILLION, OUTPUT_USD_PER_MILLION)
print(pilot_timing)
if len(pilot_rows) == 100:
    pilot_report = evaluate(pilot, pilot_rows)
    pilot_report_folder = save_reports(pilot_report, OUT / EXPERIMENT_VERSION / 'pilot')
    (pilot_report_folder / 'timing_cost.json').write_text(json.dumps(pilot_timing, indent=2))
    display(pilot_report['summary'])
    display(pilot_report['per_class'])
    display(pilot_report['disagreements'].head(20))
else:
    print(f'Complete the remaining {100-len(pilot_rows)} reviews before final pilot evaluation.')


{'successful': 100, 'failed_attempts': 0, 'wall_seconds_recorded': 50.482147875009105, 'median_request_seconds': np.float64(0.11469766707159579), 'p95_request_seconds': np.float64(0.2990940962161403), 'successful_reviews_per_minute': 118.85389692323818, 'input_tokens_known': 169697, 'output_tokens_known': 31723, 'estimated_usd_known_usage': 0.007127274, 'attempts_without_usage': 0, 'input_usd_per_million': 0.042, 'output_usd_per_million': 0.0}
Saved report: /Users/rejaimes/repos/coffee-project/data/processed/jev/jev_questions_v1/pilot/evaluation_8edebfbbaf45


,qualifier,evaluated_reviews,disagreements,agreement_pct,macro_f1_present_classes
0,good_wifi,100,0,100.0,1.000000
1,easy_parking,100,0,100.0,1.000000
2,quiet_atmosphere,100,2,98.0,0.828179
3,enough_seating,100,3,97.0,0.846446
4,outlets_available,100,0,100.0,1.000000
5,work_friendly,100,0,100.0,1.000000
6,good_drinks,100,11,89.0,0.847468
7,fun_atmosphere,100,17,83.0,0.720012


,qualifier,label,precision,recall,f1,human_count,predicted_count
0,good_wifi,-1,NaN,NaN,NaN,0,0
1,good_wifi,0,1.000000,1.000000,1.000000,99,99
2,good_wifi,1,1.000000,1.000000,1.000000,1,1
3,easy_parking,-1,1.000000,1.000000,1.000000,2,2
4,easy_parking,0,1.000000,1.000000,1.000000,97,97
5,easy_parking,1,1.000000,1.000000,1.000000,1,1
6,quiet_atmosphere,-1,NaN,NaN,NaN,0,0
7,quiet_atmosphere,0,0.979592,1.000000,0.989691,96,98
8,quiet_atmosphere,1,1.000000,0.500000,0.666667,4,2
9,enough_seating,-1,1.000000,0.800000,0.888889,5,4


,review_id,qualifier,human_label,model_label,text
0,CH2GTfItQyDiyrWqABggSA,quiet_atmosphere,1,0,Genuinely good coffee and the best I've had in...
1,KYWH59kGZ4Ao5URu8SCcQg,quiet_atmosphere,1,0,I always go here every time I head to Santa Ba...
2,7FW01ljYltmOXnOJwWXaPw,enough_seating,1,0,"I absoluely LOVE natural cafe! It is clean, q..."
3,azlzlJAXwj_ieXkvison-A,enough_seating,-1,0,we came here with a big wedding party the day ...
4,tx_ykeK9xSDIVQ-EsrhLag,enough_seating,1,0,Fabulous and hip Santa Barbara coffee shop: in...
5,RjeLE8VOJ7C03UmJCthy-w,good_drinks,1,0,Began my vegan journey early this year. So I t...
6,JxT7pEFiFNmo1HItkBes6Q,good_drinks,1,0,Here in SB from Portland OR for a girls' weeke...
7,pSMJqHIEHY6VuoH-wEowYA,good_drinks,1,0,Stopped by today from our drive from SLO to LA...
8,5Am3U4Smuncr1Zv89F3JjQ,good_drinks,-1,0,My daughter and I did not have a positive expe...
9,fbS7DEPTgiag09FAhK5fTw,good_drinks,1,0,Run don't walk to this new establishment. Brou...


## 6 — Compare Jev with existing GPT predictions (no new GPT calls)
Both models are evaluated against exactly the same corrected human labels. The old GPT pilot has a ninth social_setting field; it is excluded. GPT and Jev use different model/prompt systems, so this is a practical system comparison.

Older GPT pilot records do not have reliable request timings or token usage. Their speed and cost are **unknown**, so this notebook does not invent a speedup or cost-saving ratio. For a fair measured speed benchmark later, run GPT and Jev on the same IDs under the same concurrency/pacing and record timing and usage for both. That would require additional paid GPT requests and is not performed here.


In [6]:
if len(pilot_rows) == 100:
    gpt_rows = read_jsonl(GPT_PATH)
    gpt_report = evaluate(pilot, gpt_rows)
    comparison = pilot_report['summary'][['qualifier', 'disagreements', 'macro_f1_present_classes']].merge(
        gpt_report['summary'][['qualifier', 'disagreements', 'macro_f1_present_classes']],
        on='qualifier', suffixes=('_jev', '_gpt'), validate='one_to_one',
    )
    comparison['fewer_disagreements_with_jev'] = comparison['disagreements_gpt'] - comparison['disagreements_jev']
    display(comparison)
    comparison.to_csv(pilot_report_folder / 'jev_vs_gpt.csv', index=False)
    print('GPT historical elapsed time/cost: unknown. No benchmark ratio calculated.')
else:
    print('Finish all 100 Jev pilot predictions first.')


,qualifier,disagreements_jev,macro_f1_present_classes_jev,disagreements_gpt,macro_f1_present_classes_gpt,fewer_disagreements_with_jev
0,good_wifi,0,1.000000,0,1.000000,0
1,easy_parking,0,1.000000,0,1.000000,0
2,quiet_atmosphere,2,0.828179,3,0.661376,1
3,enough_seating,3,0.846446,3,0.868350,0
4,outlets_available,0,1.000000,0,1.000000,0
5,work_friendly,0,1.000000,2,0.663194,2
6,good_drinks,11,0.847468,4,0.917113,-7
7,fun_atmosphere,17,0.720012,6,0.897894,-11


GPT historical elapsed time/cost: unknown. No benchmark ratio calculated.


## 7 — Freeze the questions and model before holdout evaluation
Review pilot disagreements, refine questions if needed, and rerun with a **new experiment version** for each revision. Once satisfied, freeze the model, questions, and split here. Freeze uses a new file and refuses to overwrite an existing one.

Choose your acceptance criteria before opening holdout predictions. Write them below, including important positive/negative-class precision/recall needs and what to do if a class has too few examples. Do not tune rules on this holdout and then claim it is still independent. A failed evaluation calls for a new untouched test set after further development.


In [9]:
ACCEPTANCE_CRITERIA = """
Each qualifier must have at least 90% agreement with human labels.
Review positive and negative label accuracy separately.
"""
FREEZE_FOR_HOLDOUT = True
if FREEZE_FOR_HOLDOUT:
    FREEZE_FOR_HOLDOUT = False
    assert len(predictions(PILOT_RESULTS, pilot_contract)) == 100, 'Finish the pilot first.'
    assert ACCEPTANCE_CRITERIA.strip(), 'Write your holdout acceptance criteria first.'
    frozen = freeze_holdout(specs, MODEL, EXPERIMENT_VERSION, pilot, holdout_inputs, FREEZE_PATH)
    with FREEZE_PATH.with_suffix('.acceptance.txt').open('x') as file:
        file.write(ACCEPTANCE_CRITERIA)
    print('Frozen:', FREEZE_PATH)
else:
    print('Not frozen. Enable only after the pilot is complete and the rules are stable.')


Frozen: /Users/rejaimes/repos/coffee-project/data/processed/jev/frozen_jev_questions_v1.json


## 8 — Manually label the 200-review holdout, then validate it
The blank template is `data/processed/jev/holdout_human_labels.csv`. Open it in your spreadsheet app (or import into a separate Google Sheets tab), fill the eight columns with -1/0/1, then save/export back to this path. Do not fill blank cells with zero automatically. Zero means unclear/absent evidence; blank means not reviewed.

Keep IDs, business IDs, text, and the 200 selected reviews unchanged. Complete these labels **before** seeing Jev's holdout predictions. The original input-only snapshot is separate and checked below.


In [ ]:
holdout_human = pd.read_csv(HOLDOUT_HUMAN)
try:
    holdout_human = human_check(holdout_human, holdout_inputs)
    print('All 200 holdout reviews are fully labeled.')
except ValueError as exc:
    print('Not ready:', exc)


## 9 — Run the frozen model on the holdout (paid; OFF by default)
This cell checks the frozen settings and completed human labels before any paid request. It uses a separate result file, sends no human labels, and resumes only compatible successful IDs. Review-level disjointness is enforced; this is not a business-disjoint test.


In [ ]:
START_JEV_HOLDOUT = False
if START_JEV_HOLDOUT:
    START_JEV_HOLDOUT = False
    check_frozen(FREEZE_PATH, specs, MODEL, EXPERIMENT_VERSION, pilot, holdout_inputs)
    holdout_human = human_check(pd.read_csv(HOLDOUT_HUMAN), holdout_inputs)
    assert os.getenv('TYPESAFE_API_KEY'), 'Missing TypeSafe key.'
    from typesafe_sdk import TypeSafeClient, RetryPolicy
    with TypeSafeClient(model=MODEL, retry=RetryPolicy(max_retries=0), timeout=60.0) as client:
        holdout_rows = run_jev(
            holdout_human, specs, client, MODEL, EXPERIMENT_VERSION, HOLDOUT_RESULTS,
            stage='holdout', min_interval=MIN_REQUEST_INTERVAL, limit=200,
        )
else:
    print('No paid holdout requests. Complete sections 7–8 before enabling.')


## 10 — Holdout evaluation and full-run decision (local only)
Run this after all 200 predictions finish. Compare results with your prewritten acceptance criteria. Check each qualifier's precision, recall, F1, confusion counts, and class coverage. Missing rare-class examples mean performance for that class is unproven, even if agreement is high. A separate targeted challenge set can test rare qualities, but do not mix its metrics into the random holdout.

This cell never switches the full dataset to Jev automatically. Keep the existing GPT workflow/results until you decide the evidence supports switching.


In [ ]:
EVALUATE_HOLDOUT = False
if EVALUATE_HOLDOUT:
    EVALUATE_HOLDOUT = False
    check_frozen(FREEZE_PATH, specs, MODEL, EXPERIMENT_VERSION, pilot, holdout_inputs)
    holdout_human = human_check(pd.read_csv(HOLDOUT_HUMAN), holdout_inputs)
    holdout_contract = contract_for(holdout_human, specs, MODEL, EXPERIMENT_VERSION, 'holdout')
    holdout_rows = predictions(HOLDOUT_RESULTS, holdout_contract)
    holdout_report = evaluate(holdout_human, holdout_rows)
    holdout_report_folder = save_reports(holdout_report, OUT / EXPERIMENT_VERSION / 'holdout')
    display(holdout_report['summary'])
    display(holdout_report['per_class'])
    display(holdout_report['confusion'])
    holdout_timing = timing_cost(HOLDOUT_RESULTS, INPUT_USD_PER_MILLION, OUTPUT_USD_PER_MILLION)
    print(holdout_timing)
    (holdout_report_folder / 'timing_cost.json').write_text(json.dumps(holdout_timing, indent=2))
else:
    print('Enable after all holdout predictions are complete.')


## References and verification
- [TypeSafe model versions and pricing](https://docs.typesafe.ai/models)
- [Python synchronous client and retry settings](https://docs.typesafe.ai/sdk/python/api/clients/sync)
- [Response fields: answers, probabilities, confidence, usage, request ID](https://docs.typesafe.ai/sdk/python/api/types/responses)
- Human reference: [pilot data Google Sheet](https://docs.google.com/spreadsheets/d/1-PPBgDX4yogfOQ9hAR_xoceQipiQvgFgYY7pKwAN0gw/edit), snapshot fetched 2026-10-26.

Prepared without paid requests. Local mock tests cover label conversion, resume compatibility, failed-response separation, metrics, and holdout integrity. Real TypeSafe latency, cost and accuracy will appear only after you execute the guarded run cells. The SDK itself requires a Python 3.10+ kernel.
